In [ ]:
import math

lam = 5
mu = 4
n = 3
m = 5
N = n + m

P = []

# считаем P0 через нормировку
coeffs = []

for k in range(N + 1):
    if k <= n:
        coeff = (lam/mu)**k / math.factorial(k)
    else:
        coeff = (lam/mu)**n / math.factorial(n) * (lam/(n*mu))**(k-n)
    coeffs.append(coeff)

P0 = 1 / sum(coeffs)

P = [P0 * c for c in coeffs]

P

[0.2788338116044203,
 0.34854226450552533,
 0.21783891531595334,
 0.09076621471498056,
 0.0378192561312419,
 0.015758023388017458,
 0.006565843078340609,
 0.0027357679493085867,
 0.0011399033122119111]

In [ ]:
P_otk = P[-1]
P_otk

0.0011399033122119111

In [ ]:
Q = 1 - P_otk
Q

0.9988600966877881

In [ ]:
A = lam * Q
A

4.99430048343894

In [ ]:
k_avg = sum(min(k, n) * P[k] for k in range(N + 1))
k_avg

1.248575120859735

In [ ]:
t_r = sum((k - n) * P[k] for k in range(n, N + 1))
t_r

0.10567542050059255

In [ ]:
r = sum(k * P[k] for k in range(N + 1))
r

1.3542505413603276

In [ ]:
z = r / A
t_z = t_r / A

z, t_z



(0.2711592035463247, 0.02115920354632473)

In [90]:
import numpy as np
import math

lam = 5
mu = 4
n = 3
m = 5
T = 200
N = n + m

coeffs = []
for k in range(N + 1):
    if k <= n:
        coeff = (lam/mu)**k / math.factorial(k)
    else:
        coeff = (lam/mu)**n / math.factorial(n) * (lam/(n*mu))**(k-n)
    coeffs.append(coeff)

P0 = 1 / sum(coeffs)
P = [P0 * c for c in coeffs]
P_otk = P[-1]
Q = 1 - P_otk
A = lam * Q
r = sum(k * P[k] for k in range(N + 1))
t_r = sum((k - n) * P[k] for k in range(n, N + 1))
z = r / A
t_z = t_r / A


def exp_time(rate):
    return np.random.exponential(1/rate)

time = 0
queue = []
servers = []

arrivals = 0
served = 0
refused = 0

area_L = 0
area_Lq = 0
last_event_time = 0

def update_stats(current_time):
    global area_L, area_Lq, last_event_time
    dt = current_time - last_event_time
    L_current = len(servers) + len(queue)
    Lq_current = len(queue)

    area_L += L_current * dt
    area_Lq += Lq_current * dt
    last_event_time = current_time

while time < T:

    next_arrival = time + exp_time(lam)
    next_departure = min(servers) if servers else float('inf')

    if next_arrival < next_departure and next_arrival < T:
        update_stats(next_arrival)
        time = next_arrival
        arrivals += 1

        if len(servers) < n:
            servers.append(time + exp_time(mu))
        elif len(queue) < m:
            queue.append(time)
        else:
            refused += 1

    else:
        update_stats(next_departure)
        time = next_departure

        servers.remove(next_departure)
        served += 1

        if queue:
            queue.pop(0)
            servers.append(time + exp_time(mu))

update_stats(T)

#
P_otk_prac = refused / arrivals
Q_prac = 1 - P_otk_prac
A_prac = lam * Q_prac
r_prac = area_L / T
t_r_prac = area_Lq / T
z_prac = r_prac / A_prac
t_z_prac = t_r_prac / A_prac


print("=== analitic ===")
print(f"P_otk = {P_otk:.6f}")
print(f"A = {A:.6f}")
print(f"r = {r:.6f}")
print(f"t_r = {t_r:.6f}")
print(f"z = {z:.6f}")
print(f"t_z = {t_z:.6f}")

print("\n=== prac ===")
print(f"P_otk = {P_otk_prac:.6f}")
print(f"A = {A_prac:.6f}")
print(f"r = {r_prac:.6f}")
print(f"t_r = {t_r_prac:.6f}")
print(f"z = {z_prac:.6f}")
print(f"t_z = {t_z_prac:.6f}")


=== analitic ===
P_otk = 0.001140
A = 4.994300
r = 1.354251
t_r = 0.105675
z = 0.271159
t_z = 0.021159

=== prac ===
P_otk = 0.000994
A = 4.995030
r = 1.338438
t_r = 0.126554
z = 0.267954
t_z = 0.025336
